# 🎙️ Direct Gemini Audio Transcription & Insights Pipeline

This notebook provides an end-to-end workflow to:
1. **Load/Ingest** an audio file directly from Google Drive.
2. **Transcribe** the audio using the Gemini API, generating a word-for-word, speaker-wise transcript with timestamps.
3. **Save** the resulting transcript back to Google Drive.
4. **Analyze** the transcript to extract all possible key insights, questions raised, and questions generated from the conversation.
5. **Save** the insights report to Google Drive.

## Step 1: Environment Setup
Ensure the modern Google GenAI SDK and other UI helpers are installed.

In [ ]:
%%capture
# Install the modern Google GenAI SDK and other utilities
!pip install -q google-genai ipywidgets pydantic

import os
import time
import json
from google import genai
from google.colab import drive
from google.colab import userdata

## Step 2: Google Drive Setup & Audio File Selection
Mount Google Drive and specify the path to your source audio file.

In [ ]:
# 1. Mount Google Drive to load audio files and save transcripts
try:
    drive.mount('/content/drive')
    print("Google Drive successfully mounted.")
except Exception as e:
    print(f"Drive mount error: {e}")

# 2. Define path containing your source audio files
# Adjust this path as your folder structure evolves
SOURCE_FOLDER_PATH = "/content/drive/MyDrive/AnnamAI Tasks/Outreach Activity STT + Question Generation Workflow/Sample Audio Files"
print(f"Default source folder: {SOURCE_FOLDER_PATH}")

# 3. Choose the audio file
audio_filename = input("Enter the specific audio file name (e.g., MarauliKhurad3.m4a): ").strip()
drive_file_path = os.path.join(SOURCE_FOLDER_PATH, audio_filename)

if os.path.exists(drive_file_path):
    video_id = os.path.splitext(audio_filename)[0]
    print(f"\n✨ File successfully located at: {drive_file_path}")
    print(f"✨ Subfolder database key created: {video_id}")
else:
    print(f"\n❌ Error: '{audio_filename}' was not found in '{SOURCE_FOLDER_PATH}'. Please verify the path/name.")

## Step 3: Gemini Client Initialization & Model Selection
Select your Gemini model from the dropdown. Make sure you have set the secret key `GEMINI_API_KEY` in Colab's Secrets manager.

In [ ]:
from ipywidgets import Dropdown

# 1. Initialize the modern unified Gemini developer API client
try:
    client = genai.Client(api_key=userdata.get('GEMINI_API_KEY'))
    # Quick validation ping
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents='Connection successful.'
    )
    print("Gemini API connection verified.")
except Exception as e:
    print(f"Gemini API verification failed: {e}. Confirm your Colab Secrets (GEMINI_API_KEY).")

# 2. Define the Gemini models list
gemini_models = [
    'gemini-2.5-flash',
    'gemini-3.5-flash',
    'gemini-2.0-flash',
    'gemini-1.5-pro',
    'gemini-3.1-flash-lite',
]

# 3. Create interactive model dropdown
model_selector = Dropdown(
    options=gemini_models,
    value='gemini-2.5-flash',
    description='Select Model:'
)
display(model_selector)

# 4. Track changes
MODEL_NAME = model_selector.value

def on_model_change(change):
    global MODEL_NAME
    MODEL_NAME = change.new
    print(f"Selected model updated to: {MODEL_NAME}")

model_selector.observe(on_model_change, names='value')

## Step 4: Upload Audio and Generate Speaker-Wise Timestamped Transcript
This cell uploads the audio file to the Gemini File API and prompts Gemini to produce a word-for-word speaker-attributed transcript with timestamps.

In [ ]:
# 1. Direct path routing to your specified Google Drive audio file
print(f"Uploading {os.path.basename(drive_file_path)} to Gemini File API...")
audio_file = client.files.upload(file=drive_file_path)

print("Waiting for File API processing to complete...")
# Wait briefly for the file API to finish processing the state
while True:
    file_info = client.files.get(name=audio_file.name)
    if file_info.state.name == "ACTIVE":
        print("File is active and ready for model consumption.")
        break
    elif file_info.state.name == "FAILED":
        raise ValueError("File processing failed on Gemini servers.")
    else:
        print(f"Current state: {file_info.state.name}. Waiting 5 seconds...")
        time.sleep(5)

print("\n--- Transcribing Audio ---")
# 2. Call Gemini with a structured prompt requesting timestamps and speakers
transcription_prompt = """
Your task is to generate a verbatim, word-for-word text transcript of the provided audio file. 
You must attribute speech to the correct speakers (e.g., SPEAKER_00, SPEAKER_01, etc.) and include timestamps for every speech turn.

Strict rules for transcription:
1. Format each speech segment on a new line as follows:
   [MM:SS - MM:SS] SPEAKER_NAME: Spoken dialogue verbatim.
   Example:
   [00:01 - 00:08] SPEAKER_00: ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ ਜੀ।
   [00:08 - 00:12] SPEAKER_01: ਸਤਿ ਸ੍ਰੀ ਅਕਾਲ। ਆਪਣਾ ਨਾਮ ਜੀ?
2. Do not summarize or skip any spoken content. Transcribe every word including repetitions, fillers, and dialect nuances.
3. Preserve the native language (e.g. Punjabi/Gurmukhi or Hindi/Devanagari, or any code-switching) exactly as spoken.
4. Output only the transcript. Do not include introductory notes, explanations, or summaries.
"""

response = client.models.generate_content(
    model=MODEL_NAME,
    contents=[audio_file, transcription_prompt]
)

transcript_text = response.text
print("\n--- Transcript Output ---")
print(transcript_text)

## Step 5: Save Transcript to Google Drive
Save the generated speaker-attributed, timestamped transcript as a text file in Google Drive.

In [ ]:
# 1. Define output directory path in Google Drive
output_base_path = "/content/drive/MyDrive/annam AI tasks/outreach activity/transcription result/backup fallback strategies/"
target_folder = os.path.join(output_base_path, video_id)
os.makedirs(target_folder, exist_ok=True)

# 2. Define target file path
transcript_filename = f"{video_id.lower()}_transcript.txt"
transcript_path = os.path.join(target_folder, transcript_filename)

# 3. Save to file
with open(transcript_path, "w", encoding="utf-8") as f:
    f.write(transcript_text)

print(f"✨ Success! Speaker-wise, timestamped transcript saved to: {transcript_path}")

## Step 6: Extract Insights & Questions
Use Gemini to extract all possible key insights, agricultural or socioeconomic concerns, and questions (both raised in conversation and potential follow-up questions) from the generated transcript.

In [ ]:
print("\n--- Generating Insights and Questions ---")

analysis_prompt = f"""
You are an expert qualitative researcher and outreach analyst. Review the provided speaker-wise transcript.
Extract two distinct components:

1. **Key Insights & Takeaways**:
   - What are the core topics, concerns, or observations discussed?
   - Identify specific agricultural issues (e.g., crop diseases like rust, leaf folder, or issues with fertilizer shortage, electricity supply, machinery availability, seed quality).
   - Summarize the socioeconomic conditions or difficulties mentioned by the participants.

2. **Generated Questions**:
   - List all direct questions asked during the conversation.
   - List potential follow-up questions that could be asked to dive deeper into the insights/problems raised (e.g., if they mention urea shortage, what questions should we ask next to understand the supply chain bottleneck?).

Format the output in clear, structured Markdown. Use headings, bullet points, and clean formatting.

Transcript:
"""
{transcript_text}
"""
"""

analysis_response = client.models.generate_content(
    model=MODEL_NAME,
    contents=analysis_prompt
)

insights_text = analysis_response.text
print("\n--- Insights & Questions Output ---")
print(insights_text)

## Step 7: Save Insights & Questions to Google Drive
Save the generated insights and questions report as a Markdown file in Google Drive.

In [ ]:
# 1. Define target file path for insights
insights_filename = f"{video_id.lower()}_insights.md"
insights_path = os.path.join(target_folder, insights_filename)

# 2. Save insights report
with open(insights_path, "w", encoding="utf-8") as f:
    f.write(insights_text)

print(f"✨ Success! Insights and questions report saved to: {insights_path}")